# Chapter 06 데모 — 관찰성과 평가: 측정하고, 검증하고, 프로덕션으로

> 📑 대응 강의: **06-DeploymentObservablity.pptx** (프로덕션 모니터링 및 관찰성)

| 챕터 | 노트북 | 고객지원 에이전트에 추가되는 기능 |
|---|---|---|
| 02 | `02_Runtime_Demo` | Strands 에이전트 → **AgentCore Runtime** 서버리스 배포, 세션 격리, 버전/엔드포인트 |
| 03 | `03_SecurityIdentity_Demo` | **Cognito JWT 인바운드 인증**, 사용자 식별·감사 로그, **API 키 아웃바운드 인증**(배송 파트너 API) |
| 04 | `04_ToolsGateway_Demo` | **Code Interpreter**, MCP, **AgentCore Gateway**(Lambda→MCP 도구), **AgentCore Policy**(Cedar) |
| 05 | `05_Memory_Demo` | **AgentCore Memory** 단기/장기 메모리 → 세션을 넘어 고객을 기억 |
| 👉 **06** | `06_Observability_Evaluation_Demo` | **Observability**(트레이스/스팬/지표), **Evaluations**(온디맨드/온라인), PROD 승격 |
| 99 | `99_Cleanup` | 데모 리소스 일괄 삭제 |

### 지금까지 완성된 고객지원 에이전트 (DEFAULT = 최신 버전)
```
 고객 ─(Cognito JWT)─▶ AgentCore Runtime ─ 고객지원 에이전트
                        │   ├─ 로컬 도구 · 비동기 리포트 · 배송조회(Identity API 키)
                        │   ├─ Code Interpreter
                        │   ├─ Gateway(MCP) ─[Policy: Cedar]─▶ Lambda(보증·프로필·환불)
                        │   └─ Memory(단기: 대화 / 장기: 선호·사실·요약)
                        └─ OpenTelemetry ──▶ CloudWatch (GenAI Observability) ──▶ AgentCore Evaluations
```

### 이번 챕터에서 할 일
| 단계 | 내용 | 슬라이드 |
|---|---|---|
| 1 | CloudWatch **Transaction Search** 활성화 | 18 |
| 2 | 대표 고객 시나리오로 트래픽 생성 | — |
| 3 | **세션 → 트레이스 → 스팬** 분석 (콘솔 + Logs Insights) | 4–17 |
| 4 | AgentCore Runtime · Bedrock 모델 **지표** 확인 | 9–12 |
| 5 | **온디맨드 평가** — 세션/트레이스/스팬 수준 평가자 | 21–26, 29 |
| 6 | **온라인 평가** — 프로덕션 트래픽 지속 평가 | 27–28, 30 |
| 7 | 검증된 버전을 **`PROD` 엔드포인트로 승격** (02장 버전/엔드포인트의 마무리) | 13–15 |

## 0. 환경 준비

In [ ]:
import json, time
from datetime import datetime, timedelta, timezone

import boto3
from IPython.display import Markdown, display

from demo_helpers import config
from demo_helpers.aws_setup import get_region, get_account_id, wait_for, get_bearer_token, enable_transaction_search
from demo_helpers.invoke import invoke_with_jwt, new_session_id, show

REGION = get_region()
ACCOUNT_ID = get_account_id()
state = config.require_state("agent_arn", "agent_id", "memory_id", "gateway_url")
agent_arn, agent_id = state["agent_arn"], state["agent_id"]
control = boto3.client("bedrock-agentcore-control", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cloudwatch = boto3.client("cloudwatch", region_name=REGION)

alice_token = get_bearer_token("alice")
bob_token = get_bearer_token("bob")

## 1. 관찰성 준비 (슬라이드 4–7, 18)

에이전트는 모델 추론에 따라 **비결정적 제어 흐름**을 따르므로, 무엇이 왜 일어났는지 추적할 수 있어야 합니다.

| 개념 | 의미 | 이 데모에서 |
|---|---|---|
| **세션** | 사용자와의 전체 상호작용 | 런타임 세션 ID (`alice-...`) |
| **트레이스** | 요청 1건의 처리 전체 | 에이전트 호출 1회 |
| **스팬 / 서브 스팬** | 트레이스 안의 개별 작업 | 모델 호출, 도구 실행, Memory·Gateway 호출 |

**요구 사항 체크리스트 (슬라이드 18)**
- ✅ `aws-opentelemetry-distro` + `opentelemetry-instrument` 실행 → 스타터 툴킷이 Dockerfile 에 자동 적용 (02장)
- ✅ 실행 역할의 X-Ray · CloudWatch Logs 권한 → 02장에서 생성한 역할에 포함
- ⬜ 계정/리전별 **CloudWatch Transaction Search** 활성화 → 지금 실행

In [ ]:
enable_transaction_search()

## 2. 고객 시나리오로 트래픽 생성

Transaction Search 를 방금 켰다면 이전 트레이스는 인덱싱되지 않았을 수 있으므로 새 트래픽을 만듭니다.

In [ ]:
SCENARIOS = {
    "alice": (alice_token, [
        "안녕하세요! 제 고객 등급이랑 노트북 시리얼 SN1000001 보증 상태 확인해 주세요.",
        "제 취향에 맞는 헤드폰 추천해 주시고 반품 조건도 알려주세요.",
        "주문 ORD-1001 헤드폰(349,000원)을 개봉 후 반품할게요. 정책대로 환불액을 계산해서 환불 접수해 주세요.",
    ]),
    "bob": (bob_token, [
        "주문 ORD-2002 배송 어디쯤인가요?",
        "주문 ORD-2003 노트북 1,890,000원 환불해 주세요.",
    ]),
}

eval_sessions = {}
for user, (token, prompts) in SCENARIOS.items():
    sid = new_session_id(f"{user}-obs")
    eval_sessions[user] = sid
    for p in prompts:
        r = invoke_with_jwt(agent_arn, p, token, sid)
        print(f"[{user}] Q: {p[:50]}...\n        A: {r['response'][:120].strip()}...  (tools: {r.get('tools_used')})\n")

config.save_state(eval_sessions=eval_sessions)
print(eval_sessions)

## 3. 세션 → 트레이스 → 스팬 분석 (슬라이드 8, 11–17)

### 3-1. CloudWatch GenAI Observability 콘솔
아래 링크에서 **에이전트 보기 → 에이전트 세부 정보 → 세션 세부 정보 → 트레이스 세부 정보(스팬 타임라인/궤적)** 순서로 드릴다운해 보세요.

In [ ]:
console = f"https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}"
display(Markdown(f"""
- 🤖 [Bedrock AgentCore 에이전트 보기]({console}#gen-ai-observability/agent-core/agents)
- 📊 [모델 간접 호출 대시보드]({console}#gen-ai-observability/model-invocations)
- 🔎 [Transaction Search (aws/spans)]({console}#xray:traces/query)
- 📜 [런타임 로그 그룹]({console}#logsV2:log-groups/log-group/$252Faws$252Fbedrock-agentcore$252Fruntimes$252F{agent_id}-DEFAULT)

alice 세션 ID: `{eval_sessions['alice']}`
"""))

### 3-2. 코드로 스팬 조회 — CloudWatch Logs Insights (`aws/spans`)
스팬은 Transaction Search 를 통해 `aws/spans` 로그 그룹에 저장됩니다. 반영까지 **2~5분** 걸릴 수 있습니다.

In [ ]:
def logs_insights(query, log_group="aws/spans", minutes=60):
    q = logs.start_query(logGroupName=log_group, queryString=query,
                         startTime=int(time.time() - minutes * 60), endTime=int(time.time()))
    while True:
        res = logs.get_query_results(queryId=q["queryId"])
        if res["status"] in ("Complete", "Failed", "Cancelled"):
            return [{f["field"]: f["value"] for f in row} for row in res["results"]]
        time.sleep(2)

alice_sid = eval_sessions["alice"]
SPAN_QUERY = f'''
fields @timestamp, traceId, name, durationNano / 1000000 as durationMs
| filter @message like "{alice_sid}"
| sort @timestamp asc
| limit 200
'''
for i in range(20):
    spans = logs_insights(SPAN_QUERY)
    if spans:
        break
    print(f"⏳ 스팬 인덱싱 대기... ({(i + 1) * 15}s)")
    time.sleep(15)

traces = {}
for s in spans:
    traces.setdefault(s.get("traceId"), []).append(s)
print(f"alice 세션: 트레이스 {len(traces)}개, 스팬 {len(spans)}개\n")
for n, (trace_id, items) in enumerate(traces.items(), 1):
    print(f"── 트레이스 {n}: {trace_id}")
    for s in items:
        print(f"   {float(s.get('durationMs', 0)):>9.1f} ms  {s.get('name')}")

In [ ]:
# 스팬 유형별 집계 — 어디서 시간이 쓰이는가? (모델 호출 vs 도구 vs Memory/Gateway)
for row in logs_insights(f'''
filter @message like "{alice_sid}"
| stats count(*) as spans, avg(durationNano) / 1000000 as avgMs, max(durationNano) / 1000000 as maxMs by name
| sort avgMs desc
'''):
    print(f"{row.get('name', '')[:60]:60} x{row['spans']:>3}  avg {float(row['avgMs']):>8.1f} ms  max {float(row['maxMs']):>8.1f} ms")

> **사용자 지정 계측 (슬라이드 19)**: 에이전트 코드에서 OpenTelemetry API 로 비즈니스 속성을 스팬에 추가할 수 있습니다.
> ```python
> from opentelemetry import trace
> trace.get_current_span().set_attribute("customer.tier", "GOLD")
> ```

## 4. 지표 확인 (슬라이드 9–12)

In [ ]:
end = datetime.now(timezone.utc)
start = end - timedelta(hours=3)

def metric_sum(namespace, metric, dims, stat="Sum"):
    pts = cloudwatch.get_metric_statistics(Namespace=namespace, MetricName=metric, Dimensions=dims,
                                           StartTime=start, EndTime=end, Period=10800, Statistics=[stat])["Datapoints"]
    return pts[0][stat] if pts else 0

print("── AgentCore Runtime 지표 (namespace: bedrock-agentcore)")
for m in cloudwatch.list_metrics(Namespace="bedrock-agentcore")["Metrics"]:
    if any(agent_id in d["Value"] or config.AGENT_NAME in d["Value"] for d in m["Dimensions"]):
        stat = "Average" if any(k in m["MetricName"] for k in ("Latency", "Duration")) else "Sum"
        dims = ", ".join(f"{d['Name']}={d['Value'][-30:]}" for d in m["Dimensions"])
        print(f"  {m['MetricName']:28} {stat:7} {metric_sum('bedrock-agentcore', m['MetricName'], m['Dimensions'], stat):>10.1f}   [{dims}]")

print("\n── Bedrock 모델 토큰 사용량 (namespace: AWS/Bedrock)")
for metric in ("Invocations", "InputTokenCount", "OutputTokenCount"):
    for m in cloudwatch.list_metrics(Namespace="AWS/Bedrock", MetricName=metric)["Metrics"]:
        dims = {d["Name"]: d["Value"] for d in m["Dimensions"]}
        if "haiku" in dims.get("ModelId", "").lower() or "nova" in dims.get("ModelId", "").lower():
            print(f"  {metric:18} {metric_sum('AWS/Bedrock', metric, m['Dimensions']):>10.0f}   {dims.get('ModelId')}")

## 5. 온디맨드 평가 (슬라이드 21–26, 29)

AgentCore Evaluations 는 관찰성 데이터(스팬)를 입력으로 **LLM-as-a-judge** 평가를 수행합니다.

| 수준 | 평가자 예 | 질문 |
|---|---|---|
| **세션** | `Builtin.GoalSuccessRate` | 여러 턴에 걸쳐 고객의 목표를 달성했는가? |
| **트레이스** | `Builtin.Helpfulness`, `Builtin.Correctness`, `Builtin.Faithfulness` … | 각 응답이 유용하고 정확한가? |
| **스팬** | `Builtin.ToolSelectionAccuracy`, `Builtin.ToolParameterAccuracy` | 올바른 도구를 올바른 파라미터로 썼는가? |

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation(region=REGION)
EVALUATORS = ["Builtin.GoalSuccessRate", "Builtin.Helpfulness", "Builtin.Correctness", "Builtin.ToolSelectionAccuracy"]

results = None
for attempt in range(6):
    try:
        results = eval_client.run(agent_id=agent_id, session_id=eval_sessions["alice"], evaluators=EVALUATORS,
                                  output="eval_results_alice.json")
        if results.results:
            break
    except Exception as e:
        print(f"⏳ 평가 데이터 준비 대기 ({e})")
    time.sleep(30)

In [ ]:
assert results and results.results, "평가 결과가 없습니다. 몇 분 뒤 위 셀을 다시 실행하세요."
rows = ["| 평가자 | 수준 대상 | 점수 | 라벨 | 근거 (요약) |", "|---|---|---|---|---|"]
for r in results.results:
    span = r.context.get("spanContext", {}) if r.context else {}
    target = "trace " + span["traceId"][-6:] if span.get("traceId") else "session"
    explanation = (r.explanation or r.error or "").replace("\n", " ").replace("|", "/")[:160]
    rows.append(f"| {r.evaluator_name} | {target} | {r.value} | {r.label} | {explanation} |")
display(Markdown("\n".join(rows)))

👉 낮은 점수가 나온 트레이스는 3단계의 스팬 타임라인에서 **어떤 도구 선택·응답이 문제였는지** 확인해 프롬프트·도구 설명을 개선합니다.
CI/CD 에서 배포 전 이 평가를 돌려 **품질 게이트**로 쓸 수 있습니다 (슬라이드 27).

## 6. 온라인 평가 (슬라이드 27–28, 30)
프로덕션 트래픽을 **샘플링**해 지속적으로 평가합니다. (데모는 100% 샘플링, 실서비스는 보통 5–20%)

In [ ]:
try:
    online = eval_client.create_online_config(
        config_name=config.ONLINE_EVAL_CONFIG_NAME,
        agent_id=agent_id,
        sampling_rate=100,
        evaluator_list=["Builtin.GoalSuccessRate", "Builtin.Helpfulness", "Builtin.ToolSelectionAccuracy"],
        config_description="Customer support agent - continuous quality monitoring",
        auto_create_execution_role=True,
    )
    config.save_state(online_eval_config_id=online["onlineEvaluationConfigId"])
    print(json.dumps(online, indent=2, default=str)[:1500])
except Exception as e:
    print("⚠️ 생성 실패(이미 존재할 수 있음):", e)
    print(json.dumps(eval_client.list_online_configs(agent_id=agent_id), indent=2, default=str)[:1500])

In [ ]:
# 온라인 평가 대상이 될 새 트래픽 생성
for p in ["모니터 보증 기간이 얼마나 되나요?", "스마트폰 반품하려면 어떻게 해야 하나요?"]:
    show(invoke_with_jwt(agent_arn, p, alice_token, new_session_id("alice-online")))

몇 분 뒤 **CloudWatch → GenAI Observability → Bedrock AgentCore → 에이전트 → `DEFAULT` 엔드포인트 → Evaluations 탭** 에서
평가자별 평균 점수와 추이, 평가된 세션/트레이스 목록을 확인할 수 있습니다 (슬라이드 30).

## 7. 프로덕션 승격: `PROD` 엔드포인트 업데이트 (슬라이드 13–15)

02장에서 `PROD` 를 V1 에 고정해 두었습니다. 평가로 품질을 확인한 **최신 버전**으로 승격합니다.
엔드포인트 ARN 은 그대로이므로 클라이언트 변경 없이 트래픽이 새 버전으로 전환되고, 문제가 생기면 이전 버전으로 즉시 롤백할 수 있습니다.

In [ ]:
latest_version = control.get_agent_runtime(agentRuntimeId=agent_id)["agentRuntimeVersion"]
prod = control.get_agent_runtime_endpoint(agentRuntimeId=agent_id, endpointName="PROD")
print(f"PROD: V{prod.get('liveVersion')} → V{latest_version} 승격")

control.update_agent_runtime_endpoint(agentRuntimeId=agent_id, endpointName="PROD", agentRuntimeVersion=latest_version)
wait_for(lambda: control.get_agent_runtime_endpoint(agentRuntimeId=agent_id, endpointName="PROD")["status"],
         ready={"READY"}, failed={"UPDATE_FAILED"}, label="PROD 엔드포인트")
config.save_state(prod_version=latest_version)

for ep in control.list_agent_runtime_endpoints(agentRuntimeId=agent_id).get("runtimeEndpoints", []):
    print(f"{ep.get('name'):8} → liveVersion={ep.get('liveVersion')}")

In [ ]:
# 이제 PROD 엔드포인트도 JWT 인증 + 모든 기능을 갖춘 최종 에이전트를 서비스
show(invoke_with_jwt(agent_arn, "안녕하세요, 저 기억하세요? 오늘은 주문 ORD-1001 배송 상태만 확인해 주세요.",
                     alice_token, new_session_id("alice-prod"), qualifier="PROD"))

> 롤백도 같은 API 로: `control.update_agent_runtime_endpoint(agentRuntimeId=agent_id, endpointName="PROD", agentRuntimeVersion="<이전 버전>")`

## 🎉 전체 시나리오 완료

| 챕터 | AgentCore 서비스 | 고객지원 에이전트가 얻은 능력 |
|---|---|---|
| 02 | **Runtime** | 서버리스 배포, 세션별 microVM 격리, 비동기 작업, 버전/엔드포인트 |
| 03 | **Identity** (+ Cognito) | 로그인 고객만 호출, 사용자 컨텍스트·감사 로그, 외부 API 키 안전 사용 |
| 04 | **Code Interpreter · Gateway · Policy** | 정확한 계산, 사내 Lambda 를 공유 MCP 도구로, 환불 한도 강제 |
| 05 | **Memory** | 대화 저장·복원, 고객 선호·사실 기억, 고객 간 기억 격리 |
| 06 | **Observability · Evaluations** | 트레이스/지표로 동작 가시화, 품질 평가 후 PROD 승격 |

➡️ 데모를 마친 뒤에는 **[99_Cleanup](99_Cleanup.ipynb)** 으로 리소스를 삭제하세요 (온라인 평가는 실행되는 동안 비용이 발생합니다).